# Academic Risk Prediction: Final Consolidated Run

One notebook that produces every table and figure needed for the Results section, using **group-aware validation** throughout.

**What it does**
1. Loads the data, builds the target (`CGPA < 3.00`), and audits duplicates (reconciles the 680 vs 605 counts).
2. Compares **raw vs recoded sleep time** (sleep clock hours are recoded to hours after 18:00).
3. Runs 6 single models + dummy (stacking uses 4 base models: RF, ET, HGB, XGB) under **random split (leaky reference)** and **group-aware CV**, repeated over seeds.
4. Runs a **nested group-aware stacking ensemble** with a training-only F1 threshold.
5. Adds **group-bootstrap 95% CIs**, a calibration plot, and a **label-permutation null test**.
6. Computes **permutation importance and SHAP on the group-aware Extra Trees model**, averaged over folds.
7. Records hyperparameters and library versions for the paper, and zips everything in `outputs/`.

**How to use in Colab:** Runtime → Run all. When asked, upload `audited_dataset.csv` (from your `academic_risk_outputs.zip`) or the original `data.csv` / `data.xlsx`.
Reduce `SEEDS`, `N_TREES`, `N_BOOT`, `N_PERM_NULL` in the config cell for a quick trial run.

In [ ]:
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
try:
    import shap
except Exception:
    _pip("shap")
try:
    import xgboost
except Exception:
    _pip("xgboost")

## Methodological corrections in this version
- The default input is `data updated.csv` unless `DATA_PATH` is explicitly set.
- Group-bootstrap confidence intervals are computed from fold-specific metrics, then averaged, rather than pooling predictions across fold-specific models.
- The stacking threshold is selected from group-aware cross-fitted meta-model predictions, avoiding threshold selection on in-sample meta-model predictions.
- Re-run the entire notebook to regenerate metrics, figures, and the ZIP; old outputs are not valid substitutes for the corrected run.


In [ ]:
import os, json, warnings, time, shutil, platform
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import sklearn, xgboost, shap
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss, accuracy_score,
                             balanced_accuracy_score, precision_score, recall_score, f1_score)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

# ------------------------------ CONFIG ------------------------------
CFG = dict(
    SEEDS=[0, 1, 2, 3, 4],        # repeated CV seeds for model comparison
    N_SPLITS=5,
    N_TREES=500,                  # RF / ET trees
    RUN_STACK=True,
    STACK_SEEDS=[0, 1, 2],
    N_PERM_NULL=50,               # label-permutation runs
    N_BOOT=1000,                  # group-bootstrap resamples
    XAI_SEEDS=[0, 1, 2],          # seeds for permutation importance
    SHAP_SEED=0,                  # SHAP computed for this seed only (slow)
    SHAP_BACKGROUND=100,
    SHAP_ROWS_PER_FOLD=150,
    PERM_REPEATS=10,
    PRIMARY_MODEL="Extra Trees",
    TARGET_THRESHOLD=3.00,
)
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
print({k: v for k, v in CFG.items()})

## 1. Load data and build the target

> **Updated dataset note:** This notebook now selects `data updated.csv` by default. If you set the environment variable `DATA_PATH`, that explicit path takes priority. Check the printed `Using:` line before running the analysis.


In [ ]:
CANDIDATES = [os.environ.get("DATA_PATH", "data updated.csv"), "data updated.csv",
              "/content/audited_dataset.csv", "/content/data.csv", "/content/data.xlsx",
              "audited_dataset.csv", "data.csv", "data.xlsx"]
path = next((p for p in CANDIDATES if p and os.path.exists(p)), None)
if path is None:
    try:
        from google.colab import files
        up = files.upload()
        path = list(up.keys())[0]
    except Exception as e:
        raise FileNotFoundError("Upload audited_dataset.csv, data.csv or data.xlsx to the working folder.") from e
print("Using:", path)
raw = pd.read_excel(path) if path.endswith((".xlsx", ".xls")) else pd.read_csv(path, encoding="utf-8", encoding_errors="ignore")
raw.columns = [" ".join(str(c).split()) for c in raw.columns]          # collapse double spaces
raw = raw.drop(columns=[c for c in raw.columns if c.lower().startswith("unnamed")], errors="ignore")
cgpa_col = next(c for c in raw.columns if "cgpa" in c.lower())
y = (raw[cgpa_col] < CFG["TARGET_THRESHOLD"]).astype(int).to_numpy()
X_raw = raw.drop(columns=[cgpa_col] + [c for c in raw.columns if c.lower() == "academic_risk"])
print("Rows:", len(X_raw), "| predictors:", X_raw.shape[1], "| at-risk:", y.sum(), f"({y.mean()*100:.2f}%)")
print("Missing cells:", int(X_raw.isna().sum().sum()))
X_raw.dtypes.to_frame("dtype").T

## 2. Duplicate audit (reconciles 680 vs 605)

In [ ]:
pattern_key = X_raw.astype(str).agg("|".join, axis=1)
groups = pd.factorize(pattern_key)[0]
n = len(X_raw)
rows_dup_feat_and_target = int(pd.concat([X_raw, pd.Series(y, name="y")], axis=1).duplicated().sum())
rows_dup_feat_only = int(X_raw.duplicated().sum())
rows_dup_feat_cgpa = int(pd.concat([X_raw, raw[cgpa_col]], axis=1).duplicated().sum())
g = pd.DataFrame({"g": groups, "y": y}).groupby("g")["y"].agg(["size", "nunique"])
conflict_groups = int((g["nunique"] > 1).sum())
conflict_rows = int(g.loc[g["nunique"] > 1, "size"].sum())
audit = {
    "rows": n,
    "exact_duplicate_rows (features+label)": rows_dup_feat_and_target,
    "distinct_records (features+label)": n - rows_dup_feat_and_target,
    "exact_duplicate_rows (features + exact CGPA)  <- the 309 / 680 in Experiment E": rows_dup_feat_cgpa,
    "distinct_records (features + exact CGPA)": n - rows_dup_feat_cgpa,
    "exact_duplicate_rows (features only)": rows_dup_feat_only,
    "unique_answer_patterns (features only) = groups": int(g.shape[0]),
    "patterns_with_conflicting_labels": conflict_groups,
    "rows_in_conflicting_patterns": conflict_rows,
}
audit_df = pd.Series(audit, name="value").to_frame()
audit_df.to_csv(f"{OUT}/01_duplicate_audit.csv")
audit_df

## 3. Feature sets: raw vs recoded sleep time
The sleep column mixes after-midnight hours (0 to 8) and evening hours (22, 23). The recoded version uses **hours after 18:00**, `(h - 18) mod 24`, so 22→4, 23→5, 0→6, 1→7, ...

In [ ]:
sleep_col = next(c for c in X_raw.columns if "go to sleep" in c.lower())
print("Sleep column:", sleep_col, "| values:", sorted(X_raw[sleep_col].unique()))
X_variants = {"raw_sleep": X_raw.copy(), "recoded_sleep": X_raw.copy()}
X_variants["recoded_sleep"][sleep_col] = (X_raw[sleep_col].astype(float) - 18) % 24
chk = pd.DataFrame({"sleep_raw": X_raw[sleep_col], "hours_after_18": X_variants["recoded_sleep"][sleep_col], "risk": y})
sleep_tab = chk.groupby("hours_after_18").agg(n=("risk", "size"), at_risk_rate=("risk", "mean")).round(3)
sleep_tab.to_csv(f"{OUT}/02_sleep_recoded_risk_rate.csv")
sleep_tab

## 4. Models and preprocessing (all fitted inside training folds)

In [ ]:
def split_cols(X):
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat = [c for c in X.columns if c not in num]
    return num, cat

def make_prep(X, scale):
    num, cat = split_cols(X)
    return ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])

def make_estimator(name, seed):
    T = CFG["N_TREES"]
    return {
        "Dummy (prior)": DummyClassifier(strategy="prior"),
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=seed),
        "Random Forest": RandomForestClassifier(n_estimators=T, n_jobs=-1, random_state=seed),
        "Extra Trees": ExtraTreesClassifier(n_estimators=T, n_jobs=-1, random_state=seed),
        "Gradient Boosting": GradientBoostingClassifier(random_state=seed),
        "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=seed),
        "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8,
                                 colsample_bytree=0.8, eval_metric="logloss", random_state=seed, n_jobs=-1),
    }[name]

def make_pipe(name, X, seed):
    return Pipeline([("prep", make_prep(X, scale=(name == "Logistic Regression"))),
                     ("model", make_estimator(name, seed))])

BASE = ["Logistic Regression", "Random Forest", "Extra Trees", "Gradient Boosting", "Hist Gradient Boosting", "XGBoost"]
ALL = ["Dummy (prior)"] + BASE
# Supervisor request: stack only 4 base models
STACK_BASE = ["Random Forest", "Extra Trees", "Hist Gradient Boosting", "XGBoost"]

def metrics(yt, p, thr=0.5):
    pred = (p >= thr).astype(int)
    ok = len(np.unique(yt)) == 2
    return dict(roc_auc=roc_auc_score(yt, p) if ok else np.nan,
                pr_auc=average_precision_score(yt, p) if ok else np.nan,
                brier=brier_score_loss(yt, np.clip(p, 0, 1)),
                accuracy=accuracy_score(yt, pred), balanced_acc=balanced_accuracy_score(yt, pred),
                precision=precision_score(yt, pred, zero_division=0),
                recall=recall_score(yt, pred, zero_division=0), f1=f1_score(yt, pred, zero_division=0))

def splitter(scheme, seed):
    if scheme == "group":
        return StratifiedGroupKFold(n_splits=CFG["N_SPLITS"], shuffle=True, random_state=seed)
    return StratifiedKFold(n_splits=CFG["N_SPLITS"], shuffle=True, random_state=seed)

# hyperparameter table for the paper
hp = []
for nm in ALL:
    est = make_estimator(nm, 0)
    keep = {k: v for k, v in est.get_params().items() if k in
            ["n_estimators", "max_depth", "learning_rate", "max_iter", "C", "penalty", "min_samples_leaf", "min_samples_split",
             "max_features", "subsample", "colsample_bytree", "class_weight", "strategy", "criterion", "random_state"]}
    hp.append({"model": nm, "hyperparameters": json.dumps(keep, default=str)})
hp_df = pd.DataFrame(hp); hp_df.to_csv(f"{OUT}/03_hyperparameters.csv", index=False)
hp_df

## 5. Repeated CV: random split (leaky reference) vs group-aware

In [ ]:
SCENARIOS = [("random", "raw_sleep"), ("group", "raw_sleep"), ("group", "recoded_sleep")]
rows, oof = [], {}
t0 = time.time()
for scheme, variant in SCENARIOS:
    X = X_variants[variant]
    for seed in CFG["SEEDS"]:
        for name in ALL:
            oof[(scheme, variant, name, seed)] = np.full(n, np.nan)
        for fold, (tr, te) in enumerate(splitter(scheme, seed).split(X, y, groups)):
            for name in ALL:
                pipe = make_pipe(name, X, seed).fit(X.iloc[tr], y[tr])
                p = pipe.predict_proba(X.iloc[te])[:, 1]
                oof[(scheme, variant, name, seed)][te] = p
                rows.append(dict(scheme=scheme, variant=variant, model=name, seed=seed, fold=fold, **metrics(y[te], p)))
    print(f"{scheme:6s} {variant:14s} done  ({time.time()-t0:.0f}s)")
cv = pd.DataFrame(rows); cv.to_csv(f"{OUT}/04_cv_fold_metrics.csv", index=False)

def mean_sd(df, cols=("roc_auc", "pr_auc", "brier", "recall", "precision", "f1", "balanced_acc")):
    per_seed = df.groupby(["scheme", "variant", "model", "seed"])[list(cols)].mean().reset_index()
    g = per_seed.groupby(["scheme", "variant", "model"])[list(cols)]
    m, s = g.mean(), g.std()
    out = m.copy().astype(object)
    for c in cols: out[c] = [f"{a:.3f} ± {b:.3f}" for a, b in zip(m[c], s[c])]
    return out.reset_index(), m.reset_index()
summary_txt, summary_num = mean_sd(cv)
summary_txt.to_csv(f"{OUT}/05_cv_summary_mean_sd.csv", index=False)
summary_txt[["scheme", "variant", "model", "roc_auc", "pr_auc", "recall", "precision", "f1"]]

In [ ]:
# Leakage effect and sleep recoding effect (ROC-AUC, mean over seeds)
piv = summary_num.pivot_table(index="model", columns=["scheme", "variant"], values="roc_auc").round(3)
piv.to_csv(f"{OUT}/06_auc_random_vs_group_vs_recoded.csv")
print("Prevalence (PR-AUC chance level):", round(y.mean(), 3))
piv

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
models_plot = BASE
w = 0.27
for i, (sc, va, lab) in enumerate([("random", "raw_sleep", "Random split (leaky)"),
                                   ("group", "raw_sleep", "Group split"),
                                   ("group", "recoded_sleep", "Group split + recoded sleep")]):
    vals = [summary_num.query("scheme==@sc and variant==@va and model==@m")["roc_auc"].iloc[0] for m in models_plot]
    ax.bar(np.arange(len(models_plot)) + (i - 1) * w, vals, w, label=lab)
ax.axhline(0.5, color="k", lw=0.8, ls="--")
ax.set_xticks(range(len(models_plot))); ax.set_xticklabels(models_plot, rotation=25, ha="right")
ax.set_ylabel("ROC-AUC (mean over seeds)"); ax.set_ylim(0.4, 1.0); ax.legend(frameon=False, fontsize=8)
ax.set_title("Random split overstates performance")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_auc_random_vs_group.png", dpi=200); plt.show()

### Choose the final feature set
The notebook picks whichever sleep encoding gives the higher mean ROC-AUC for the primary model under group CV. Check this yourself and report the choice in the paper.

In [ ]:
q = summary_num[(summary_num.scheme == "group") & (summary_num.model == CFG["PRIMARY_MODEL"])].set_index("variant")["roc_auc"]
FINAL_VARIANT = q.idxmax()
print(q.round(4).to_dict(), "->", FINAL_VARIANT)
X_final = X_variants[FINAL_VARIANT]

## 6. Group-bootstrap 95% CIs and calibration (final variant, seed-averaged out-of-fold probabilities)

In [ ]:
# Group bootstrap over held-out groups, keeping metric estimates fold-specific.
# This avoids pooling scores from different fold-specific models into one ROC-AUC,
# which can distort the Dummy(prior) baseline when fold prevalences differ.
rng = np.random.default_rng(42)
ci_rows = []
group_splits = {}
for seed in CFG["SEEDS"]:
    group_splits[seed] = list(splitter("group", seed).split(X_final, y, groups))

for name in ALL:
    fold_rows = cv.query(
        "scheme == 'group' and variant == @FINAL_VARIANT and model == @name"
    )
    row = {"model": name}
    for metric in ["roc_auc", "pr_auc", "brier"]:
        point = float(fold_rows[metric].mean())
        boot_values = []
        for _ in range(CFG["N_BOOT"]):
            metric_values = []
            for seed in CFG["SEEDS"]:
                pred_all = oof[("group", FINAL_VARIANT, name, seed)]
                for tr, te in group_splits[seed]:
                    fold_groups = np.unique(groups[te])
                    sampled_groups = rng.choice(
                        fold_groups, size=len(fold_groups), replace=True
                    )
                    sampled_indices = np.concatenate([
                        te[groups[te] == gid] for gid in sampled_groups
                    ])
                    yt = y[sampled_indices]
                    pp = pred_all[sampled_indices]
                    if metric in ("roc_auc", "pr_auc") and len(np.unique(yt)) < 2:
                        continue
                    if metric == "roc_auc":
                        val = roc_auc_score(yt, pp)
                    elif metric == "pr_auc":
                        val = average_precision_score(yt, pp)
                    else:
                        val = brier_score_loss(yt, np.clip(pp, 0, 1))
                    metric_values.append(val)
            if metric_values:
                boot_values.append(float(np.mean(metric_values)))
        lo, hi = np.percentile(boot_values, [2.5, 97.5])
        row[metric] = f"{point:.3f} [{lo:.3f}, {hi:.3f}]"
    ci_rows.append(row)

ci = pd.DataFrame(ci_rows)
ci.to_csv(f"{OUT}/07_group_bootstrap_95CI.csv", index=False)
ci

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4.2))
for name in [CFG["PRIMARY_MODEL"], "Random Forest", "Logistic Regression"]:
    fr, mp = calibration_curve(y, P[name], n_bins=8, strategy="quantile")
    ax.plot(mp, fr, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("Mean predicted risk"); ax.set_ylabel("Observed at-risk rate"); ax.legend(frameon=False, fontsize=8)
ax.set_title("Calibration (group-aware OOF)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_calibration.png", dpi=200); plt.show()

## 7. Label-permutation null (is the signal above chance?)

In [ ]:
null_aucs = []
for r in range(CFG["N_PERM_NULL"]):
    yp = np.random.default_rng(1000 + r).permutation(y)
    fold_auc = []
    for tr, te in splitter("group", r).split(X_final, yp, groups):
        pipe = make_pipe(CFG["PRIMARY_MODEL"], X_final, r).fit(X_final.iloc[tr], yp[tr])
        p = pipe.predict_proba(X_final.iloc[te])[:, 1]
        if len(np.unique(yp[te])) == 2: fold_auc.append(roc_auc_score(yp[te], p))
    null_aucs.append(np.mean(fold_auc))
obs = cv[(cv.scheme == "group") & (cv.variant == FINAL_VARIANT) & (cv.model == CFG["PRIMARY_MODEL"])].groupby("seed")["roc_auc"].mean().mean()
p_val = (1 + np.sum(np.array(null_aucs) >= obs)) / (1 + len(null_aucs))
pd.DataFrame({"null_auc": null_aucs}).to_csv(f"{OUT}/08_permutation_null.csv", index=False)
print(f"Observed ROC-AUC {obs:.3f} | null mean {np.mean(null_aucs):.3f} (SD {np.std(null_aucs):.3f}) | empirical p = {p_val:.3f}")
plt.figure(figsize=(4.6, 3.4)); plt.hist(null_aucs, bins=10, alpha=.7); plt.axvline(obs, color="r", label="observed")
plt.xlabel("ROC-AUC"); plt.legend(frameon=False); plt.title("Permutation null"); plt.tight_layout()
plt.savefig(f"{OUT}/fig_permutation_null.png", dpi=200); plt.show()

## 8. Nested group-aware stacking ensemble
Base models: LR, RF, ET, GB, HGB, XGB. Meta-learner: Logistic Regression (C = 0.5) on out-of-fold logits. The threshold is chosen by F1 on the **training part only** (inner OOF).

In [ ]:
def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))

THR_GRID = np.linspace(0.05, 0.95, 91)

def run_stack(seed):
    out = []
    for fold, (tr, te) in enumerate(splitter("group", seed).split(X_final, y, groups)):
        Xtr, ytr, gtr = X_final.iloc[tr], y[tr], groups[tr]
        inner = StratifiedGroupKFold(
            n_splits=CFG["N_SPLITS"], shuffle=True, random_state=seed
        )
        Z_oof = np.zeros((len(tr), len(STACK_BASE)))
        Z_te = np.zeros((len(te), len(STACK_BASE)))

        # First-level out-of-fold predictions for meta-training data.
        for j, name in enumerate(STACK_BASE):
            for itr, iva in inner.split(Xtr, ytr, gtr):
                m = make_pipe(name, X_final, seed).fit(Xtr.iloc[itr], ytr[itr])
                Z_oof[iva, j] = m.predict_proba(Xtr.iloc[iva])[:, 1]
            m = make_pipe(name, X_final, seed).fit(Xtr, ytr)
            Z_te[:, j] = m.predict_proba(X_final.iloc[te])[:, 1]

        # Choose the threshold from cross-fitted meta-model predictions,
        # not in-sample predictions from the meta-model fitted on all Z_oof.
        meta_cv = StratifiedGroupKFold(
            n_splits=CFG["N_SPLITS"], shuffle=True, random_state=seed + 1000 + fold
        )
        p_meta_cv = np.full(len(ytr), np.nan)
        for mtr, mva in meta_cv.split(Z_oof, ytr, gtr):
            meta_fold = LogisticRegression(C=0.5, max_iter=2000)
            meta_fold.fit(logit(Z_oof[mtr]), ytr[mtr])
            p_meta_cv[mva] = meta_fold.predict_proba(logit(Z_oof[mva]))[:, 1]

        valid = np.isfinite(p_meta_cv)
        f1s = [
            f1_score(ytr[valid], (p_meta_cv[valid] >= t).astype(int), zero_division=0)
            for t in THR_GRID
        ]
        tau = THR_GRID[int(np.argmax(f1s))]

        # Final meta-model is fit on all first-level OOF predictions.
        meta = LogisticRegression(C=0.5, max_iter=2000)
        meta.fit(logit(Z_oof), ytr)
        p_stack = meta.predict_proba(logit(Z_te))[:, 1]
        p_blend = Z_te.mean(axis=1)

        out.append(dict(seed=seed, fold=fold, model="Mean blend",
                        threshold=0.5, **metrics(y[te], p_blend)))
        out.append(dict(seed=seed, fold=fold, model="Stack (default 0.5)",
                        threshold=0.5, **metrics(y[te], p_stack)))
        out.append(dict(seed=seed, fold=fold, model="Stack (cross-fitted F1 threshold)",
                        threshold=tau, **metrics(y[te], p_stack, tau)))
    return out

if CFG["RUN_STACK"]:
    stack_rows = []
    for s in CFG["STACK_SEEDS"]:
        stack_rows += run_stack(s)
        print("stack seed", s, "done")
    stk = pd.DataFrame(stack_rows)
    stk.to_csv(f"{OUT}/09_stacking_fold_metrics.csv", index=False)
    per_seed = stk.groupby(["model", "seed"]).mean(numeric_only=True).reset_index()
    cols = ["roc_auc", "pr_auc", "brier", "recall", "precision", "f1", "balanced_acc", "threshold"]
    g_ = per_seed.groupby("model")[cols]
    stk_summary = pd.DataFrame(
        {c: [f"{a:.3f} ± {b:.3f}" for a, b in zip(g_.mean()[c], g_.std()[c])]
         for c in cols},
        index=g_.mean().index
    )
    stk_summary.to_csv(f"{OUT}/10_stacking_summary.csv")
    display(stk_summary)

## 9. Explainability on the group-aware Extra Trees model
Permutation importance (drop in average precision) is averaged over folds and seeds. SHAP (TreeExplainer, interventional, 100-row background) is computed for one seed; one-hot columns are summed back to the 19 original questions *per row* before taking the mean absolute value.

In [ ]:
def orig_feature_map(prep, X):
    num, cat = split_cols(X)
    mapping = []
    for fn in prep.get_feature_names_out():
        if fn.startswith("num__"): mapping.append(fn[5:])
        else:
            core = fn[5:]
            mapping.append(max([c for c in cat if core.startswith(c + "_")], key=len))
    return np.array(mapping)

def shap_pos(explainer, A):
    sv = explainer.shap_values(A, check_additivity=False)
    if isinstance(sv, list): sv = sv[1]
    sv = np.asarray(sv)
    return sv[:, :, 1] if sv.ndim == 3 else sv

perm_rows, shap_rows, pooled_sv, pooled_A, feat_names = [], [], [], [], None
for seed in CFG["XAI_SEEDS"]:
    for fold, (tr, te) in enumerate(splitter("group", seed).split(X_final, y, groups)):
        pipe = make_pipe(CFG["PRIMARY_MODEL"], X_final, seed).fit(X_final.iloc[tr], y[tr])
        r = permutation_importance(pipe, X_final.iloc[te], y[te], scoring="average_precision",
                                   n_repeats=CFG["PERM_REPEATS"], random_state=seed)
        for f, v in zip(X_final.columns, r.importances_mean):
            perm_rows.append(dict(seed=seed, fold=fold, feature=f, importance=v))
        if seed == CFG["SHAP_SEED"]:
            prep, model = pipe.named_steps["prep"], pipe.named_steps["model"]
            Atr, Ate = prep.transform(X_final.iloc[tr]), prep.transform(X_final.iloc[te])
            rs = np.random.default_rng(seed + fold)
            bg = Atr[rs.choice(len(Atr), size=min(CFG["SHAP_BACKGROUND"], len(Atr)), replace=False)]
            sel = rs.choice(len(Ate), size=min(CFG["SHAP_ROWS_PER_FOLD"], len(Ate)), replace=False)
            expl = shap.TreeExplainer(model, data=bg, feature_perturbation="interventional")
            sv = shap_pos(expl, Ate[sel])
            om = orig_feature_map(prep, X_final)
            per_row = pd.DataFrame(sv).T.groupby(om).sum().T          # rows x original features
            for f in per_row.columns:
                shap_rows.append(dict(fold=fold, feature=f, mean_abs_shap=per_row[f].abs().mean()))
            pooled_sv.append(sv); pooled_A.append(Ate[sel]); feat_names = prep.get_feature_names_out()
    print("XAI seed", seed, "done")

perm = pd.DataFrame(perm_rows).groupby("feature")["importance"].agg(["mean", "std"]).sort_values("mean", ascending=False)
shp = pd.DataFrame(shap_rows).groupby("feature")["mean_abs_shap"].agg(["mean", "std"]).sort_values("mean", ascending=False)
perm.to_csv(f"{OUT}/11_permutation_importance_group_aware.csv"); shp.to_csv(f"{OUT}/12_shap_importance_group_aware.csv")
rho = spearmanr(perm["mean"].reindex(shp.index), shp["mean"]).correlation
print(f"Spearman rank agreement between permutation and SHAP: {rho:.2f}")
pd.concat({"permutation": perm.head(10), "SHAP": shp.head(10)}, axis=1).round(4)

In [ ]:
short = lambda s: (s[:48] + "…") if len(s) > 48 else s
fig, axs = plt.subplots(1, 2, figsize=(12, 5.2))
p_ = perm.head(12)[::-1]; axs[0].barh([short(i) for i in p_.index], p_["mean"], xerr=p_["std"], color="#4C72B0")
axs[0].set_title("Permutation importance (Δ average precision)")
s_ = shp.head(12)[::-1]; axs[1].barh([short(i) for i in s_.index], s_["mean"], xerr=s_["std"], color="#DD8452")
axs[1].set_title("Mean |SHAP| (original features)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_importance_group_aware.png", dpi=200); plt.show()

SV, AA = np.vstack(pooled_sv), np.vstack(pooled_A)
shap.summary_plot(SV, AA, feature_names=[short(f.split("__", 1)[-1]) for f in feat_names], max_display=15, show=False)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_shap_beeswarm.png", dpi=200, bbox_inches="tight"); plt.show()

## 10. Library versions, summary and download

In [ ]:
versions = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, scikit_learn=sklearn.__version__,
                xgboost=xgboost.__version__, shap=shap.__version__, final_variant=FINAL_VARIANT, config={k: str(v) for k, v in CFG.items()})
json.dump(versions, open(f"{OUT}/00_versions_and_config.json", "w"), indent=2)
print(json.dumps(versions, indent=2))
shutil.make_archive("academic_risk_final_outputs", "zip", OUT)
print("Saved:", sorted(os.listdir(OUT)))
try:
    from google.colab import files; files.download("academic_risk_final_outputs.zip")
except Exception:
    print("Zip created: academic_risk_final_outputs.zip")